In [1]:
from pathlib import Path
import math
import time
import json
import shutil
import random
import zipfile
import torch
import numpy as np

import cv2
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

try:
    from skimage.metrics import structural_similarity as ssim_metric
    SKIMAGE_AVAILABLE = True
except Exception:
    SKIMAGE_AVAILABLE = False
    ssim_metric = None


In [2]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()

WORK_ROOT = PROJECT_ROOT / "vsr_workspace"
DATA_ROOT = WORK_ROOT / "data"

TRAIN_ZIP_PATH = PROJECT_ROOT / "train_sharp.zip"
VAL_ZIP_PATH = PROJECT_ROOT / "val_sharp.zip"

TRAIN_SHARP_DIR = DATA_ROOT / "train_sharp"
VAL_SHARP_DIR = DATA_ROOT / "val_sharp"

EXPERIMENT_ROOT = WORK_ROOT / "experiments" / "model_v3_seq15"
SAVE_DIR = EXPERIMENT_ROOT / "checkpoints"
EPOCH_CKPT_DIR = SAVE_DIR / "epoch_snapshots"
RESULTS_DIR = EXPERIMENT_ROOT / "results"
LOGS_DIR = EXPERIMENT_ROOT / "logs"

SPLIT_ROOT = WORK_ROOT / "experiments" / "model_v3_seq15" / "splits"

for p in [WORK_ROOT, DATA_ROOT, EXPERIMENT_ROOT, SAVE_DIR, EPOCH_CKPT_DIR, RESULTS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# runtime
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = True
USE_AMP = False 
# recipe
SEQ_LEN = 15
SCALE = 4
PATCH_SIZE = 64

BATCH_SIZE = 4
NUM_EPOCHS = 150
LR = 2e-4
NUM_WORKERS = 8
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

# split / dataset
VAL_TO_VAL_COUNT = 5
VAL_TO_TEST_COUNT = 5
MAX_TRAIN_SEQS = None
MAX_FRAMES_PER_SEQ = 200
RESET_PROCESSED_DATASET = False
FORCE_REBUILD_SPLITS = False

# scheduler / early stopping
USE_SCHEDULER = True
SCHEDULER_PATIENCE = 5
SCHEDULER_FACTOR = 0.5
MIN_LR = 1e-6

USE_EARLY_STOPPING = False
EARLY_STOPPING_PATIENCE = 12
EARLY_STOPPING_MIN_DELTA = 1e-4
EARLY_STOPPING_MIN_EPOCHS = 150

# validation
FAST_VAL_MAX_BATCHES = 20
FULL_VAL_EVERY = 2
FULL_FRAME_EVAL = False
SHAVE_BORDER = SCALE

# persistence
AUTO_RESUME = True
SAVE_PER_CLIP_METRICS = True
KEEP_EVERY_N_EPOCHS = 10

BEST_PATH = SAVE_DIR / "vsr_model_best_loss.pth"
BEST_PSNR_PATH = SAVE_DIR / "vsr_model_best_psnr.pth"
LAST_PATH = SAVE_DIR / "vsr_model_last.pth"
TRAINING_STATE_PATH = SAVE_DIR / "training_state.pth"
HISTORY_JSON_PATH = LOGS_DIR / "history.json"
TEST_METRICS_JSON_PATH = LOGS_DIR / "test_metrics.json"
TEST_PER_CLIP_JSON_PATH = LOGS_DIR / "test_per_clip_metrics.json"

FINETUNE_FROM_BEST = False
BEST_SOURCE_PATH = None

# exports
EXPORT_DEMO_CLIP_NAME = "006"
EXPORT_FPS = 25

print("device:", device)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("SPLIT_ROOT:", SPLIT_ROOT)
print("FINETUNE_FROM_BEST:", FINETUNE_FROM_BEST)
print("AUTO_RESUME:", AUTO_RESUME)
print("NUM_EPOCHS:", NUM_EPOCHS)
print("LR:", LR)


device: cuda
PROJECT_ROOT: /tf
EXPERIMENT_ROOT: /tf/vsr_workspace/experiments/model_v3_seq15
SPLIT_ROOT: /tf/vsr_workspace/experiments/model_v3_seq15/splits
FINETUNE_FROM_BEST: False
AUTO_RESUME: True
NUM_EPOCHS: 150
LR: 0.0002


In [3]:
print("TRAIN exists:", TRAIN_SHARP_DIR.exists(), TRAIN_SHARP_DIR)
print("VAL exists:", VAL_SHARP_DIR.exists(), VAL_SHARP_DIR)
print("SPLIT_ROOT exists:", SPLIT_ROOT.exists(), SPLIT_ROOT)
# print("BEST_SOURCE_PATH exists:", BEST_SOURCE_PATH.exists(), BEST_SOURCE_PATH)
print("RESET_PROCESSED_DATASET:", RESET_PROCESSED_DATASET)
print("FORCE_REBUILD_SPLITS:", FORCE_REBUILD_SPLITS)

TRAIN exists: True /tf/vsr_workspace/data/train_sharp
VAL exists: True /tf/vsr_workspace/data/val_sharp
SPLIT_ROOT exists: True /tf/vsr_workspace/experiments/model_v3_seq15/splits
RESET_PROCESSED_DATASET: False
FORCE_REBUILD_SPLITS: False


In [4]:
def list_dirs(p):
    p = Path(p)
    return sorted([x for x in p.iterdir() if x.is_dir()]) if p.exists() else []

def resolve_sequence_root(base_dir, min_expected=1, max_depth=5):
    base_dir = Path(base_dir)

    def looks_like_sequence_dir(p):
        return p.is_dir() and len(list(p.glob("*.png"))) > 0

    current_level = [base_dir]

    for _ in range(max_depth + 1):
        next_level = []
        for candidate in current_level:
            children = list_dirs(candidate)
            if len(children) >= min_expected:
                seq_like_count = sum(looks_like_sequence_dir(ch) for ch in children)
                if seq_like_count >= min_expected:
                    return candidate
            next_level.extend(children)
        current_level = next_level

    return None

def ensure_unzipped(zip_path, extract_dir):
    zip_path = Path(zip_path)
    extract_dir = Path(extract_dir)

    if not zip_path.exists():
        print(f"Zip not found, skipping: {zip_path}")
        return

    if extract_dir.exists() and len(list(extract_dir.rglob("*.png"))) > 0:
        print(f"Already extracted: {extract_dir}")
        return

    print(f"Extracting {zip_path} -> {extract_dir}")
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

ensure_unzipped(TRAIN_ZIP_PATH, TRAIN_SHARP_DIR)
ensure_unzipped(VAL_ZIP_PATH, VAL_SHARP_DIR)

TRAIN_SHARP_DIR = resolve_sequence_root(TRAIN_SHARP_DIR, min_expected=1, max_depth=5)
VAL_SHARP_DIR = resolve_sequence_root(VAL_SHARP_DIR, min_expected=1, max_depth=5)

if TRAIN_SHARP_DIR is None:
    raise RuntimeError("Could not locate train_sharp sequence folders after extraction")
if VAL_SHARP_DIR is None:
    raise RuntimeError("Could not locate val_sharp sequence folders after extraction")

train_seq_dirs_preview = list_dirs(TRAIN_SHARP_DIR)
val_seq_dirs_preview = list_dirs(VAL_SHARP_DIR)

print("Resolved TRAIN_SHARP_DIR:", TRAIN_SHARP_DIR)
print("Resolved VAL_SHARP_DIR  :", VAL_SHARP_DIR)
print("Train sequence count:", len(train_seq_dirs_preview))
print("Val sequence count  :", len(val_seq_dirs_preview))

print("\nExample train dirs:")
for p in train_seq_dirs_preview[:5]:
    print(" -", p.name)

print("\nExample val dirs:")
for p in val_seq_dirs_preview[:5]:
    print(" -", p.name)

Already extracted: /tf/vsr_workspace/data/train_sharp
Already extracted: /tf/vsr_workspace/data/val_sharp
Resolved TRAIN_SHARP_DIR: /tf/vsr_workspace/data/train_sharp/train/train_sharp
Resolved VAL_SHARP_DIR  : /tf/vsr_workspace/data/val_sharp/val/val_sharp
Train sequence count: 240
Val sequence count  : 30

Example train dirs:
 - 000
 - 001
 - 002
 - 003
 - 004

Example val dirs:
 - 000
 - 001
 - 002
 - 003
 - 004


In [5]:
def apply_jpeg_compression(img_bgr, quality=50):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)]
    success, enc = cv2.imencode(".jpg", img_bgr, encode_param)
    if not success:
        return img_bgr
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return dec

def apply_motion_blur(img_bgr, ksize=9):
    kernel = np.zeros((ksize, ksize), dtype=np.float32)
    kernel[ksize // 2, :] = 1.0
    kernel /= kernel.sum()
    return cv2.filter2D(img_bgr, -1, kernel)

def realistic_degrade_frame(
    frame_bgr, scale=4,
    blur_prob=0.6,
    motion_blur_prob=0.15,
    noise_prob=0.6,
    jpeg_prob=0.6,
    min_jpeg_quality=35,
    max_jpeg_quality=75,
    min_noise_std=1.0,
    max_noise_std=10.0,
):
    out = frame_bgr.copy()

    if random.random() < blur_prob:
        k = random.choice([3, 5])
        sigma = random.uniform(0.5, 1.6)
        out = cv2.GaussianBlur(out, (k, k), sigma)

    if random.random() < motion_blur_prob:
        ksize = random.choice([5, 7])
        out = apply_motion_blur(out, ksize=ksize)

    if random.random() < noise_prob:
        noise_std = random.uniform(min_noise_std, max_noise_std)
        noise = np.random.normal(0, noise_std, out.shape).astype(np.float32)
        out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    if random.random() < jpeg_prob:
        quality = random.randint(min_jpeg_quality, max_jpeg_quality)
        out = apply_jpeg_compression(out, quality=quality)

    h, w = out.shape[:2]
    lr = cv2.resize(out, (w // scale, h // scale), interpolation=cv2.INTER_AREA)
    return lr

def img_to_tensor_rgb(img_rgb):
    return torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0

def tensor_to_img(t):
    img = t.detach().cpu().float().numpy().transpose(1, 2, 0)
    img = np.nan_to_num(img, nan=0.0, posinf=1.0, neginf=0.0)
    img = (np.clip(img, 0, 1) * 255.0).astype(np.uint8)
    return img

def shave_tensor_border(x, border):
    if border <= 0:
        return x
    return x[..., border:-border, border:-border]

def calc_psnr(pred, target, max_val=1.0, shave_border=0):
    if not torch.isfinite(pred).all() or not torch.isfinite(target).all():
        return float("nan")

    pred = shave_tensor_border(pred, shave_border)
    target = shave_tensor_border(target, shave_border)

    mse = F.mse_loss(pred, target).item()
    if not math.isfinite(mse):
        return float("nan")
    if mse == 0:
        return 100.0
    return 20 * math.log10(max_val / math.sqrt(mse))

def calc_ssim(pred, target, shave_border=0):
    if not SKIMAGE_AVAILABLE:
        return None
    if not torch.isfinite(pred).all() or not torch.isfinite(target).all():
        return None

    pred = shave_tensor_border(pred, shave_border)
    target = shave_tensor_border(target, shave_border)

    pred_img = tensor_to_img(pred.squeeze(0))
    target_img = tensor_to_img(target.squeeze(0))

    try:
        return ssim_metric(pred_img, target_img, channel_axis=2, data_range=255)
    except Exception:
        return None

def calc_ssim_sequence(pred_seq, target_seq, shave_border=0):
    vals = []
    t = pred_seq.shape[1]
    for i in range(t):
        s = calc_ssim(pred_seq[:, i], target_seq[:, i], shave_border=shave_border)
        if s is not None:
            vals.append(s)
    return float(np.mean(vals)) if len(vals) > 0 else None

def bicubic_upsample_sequence(lr_seq, scale):
    b, t, c, h, w = lr_seq.shape
    x = lr_seq.reshape(b * t, c, h, w)
    up = F.interpolate(x, scale_factor=scale, mode="bicubic", align_corners=False)
    up = torch.clamp(up, 0.0, 1.0)
    up = torch.nan_to_num(up, nan=0.0, posinf=1.0, neginf=0.0)
    return up.view(b, t, c, h * scale, w * scale)

def random_augment_sequences(lr_seq, hr_seq):
    if random.random() < 0.5:
        lr_seq = [np.ascontiguousarray(img[:, ::-1, :]) for img in lr_seq]
        hr_seq = [np.ascontiguousarray(img[:, ::-1, :]) for img in hr_seq]
    if random.random() < 0.5:
        lr_seq = [np.ascontiguousarray(img[::-1, :, :]) for img in lr_seq]
        hr_seq = [np.ascontiguousarray(img[::-1, :, :]) for img in hr_seq]
    if random.random() < 0.5:
        lr_seq = [np.ascontiguousarray(np.rot90(img)) for img in lr_seq]
        hr_seq = [np.ascontiguousarray(np.rot90(img)) for img in hr_seq]
    return lr_seq, hr_seq

print("Degradation + metrics helpers ready")


Degradation + metrics helpers ready


In [6]:
def split_cache_ready(root):
    root = Path(root)
    if not root.exists():
        return False

    for split in ["train", "val", "test"]:
        split_dir = root / split
        if not split_dir.exists():
            return False
        clip_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
        if len(clip_dirs) == 0:
            return False

    return True

def list_sequence_dirs(root_dir):
    root_dir = Path(root_dir)
    return sorted([p for p in root_dir.iterdir() if p.is_dir()])

def extract_from_reds_sequence(seq_dir, split_name, clip_name, scale=4, max_frames=200):
    seq_dir = Path(seq_dir)
    frame_paths = sorted(seq_dir.glob("*.png"))
    if len(frame_paths) == 0:
        raise RuntimeError(f"No PNG frames found in sequence dir: {seq_dir}")

    if max_frames is not None:
        frame_paths = frame_paths[:max_frames]

    hr_dir = SPLIT_ROOT / split_name / clip_name / "hr_frames"
    lr_dir = SPLIT_ROOT / split_name / clip_name / "lr_frames"
    hr_dir.mkdir(parents=True, exist_ok=True)
    lr_dir.mkdir(parents=True, exist_ok=True)

    count = 0
    for frame_path in frame_paths:
        frame = cv2.imread(str(frame_path), cv2.IMREAD_COLOR)
        if frame is None:
            print(f"Skipping unreadable frame: {frame_path}")
            continue

        cv2.imwrite(str(hr_dir / f"{count:04d}.png"), frame)
        lr_frame = realistic_degrade_frame(frame, scale=scale)
        cv2.imwrite(str(lr_dir / f"{count:04d}.png"), lr_frame)
        count += 1

    print(f"Built {split_name}/{clip_name} from {seq_dir.name} with {count} frames")

train_seq_dirs = list_sequence_dirs(TRAIN_SHARP_DIR)
val_seq_dirs_all = list_sequence_dirs(VAL_SHARP_DIR)

if len(train_seq_dirs) == 0:
    raise RuntimeError(f"No train sequences found in {TRAIN_SHARP_DIR}")

if len(val_seq_dirs_all) < (VAL_TO_VAL_COUNT + VAL_TO_TEST_COUNT):
    raise RuntimeError(
        f"Need at least {VAL_TO_VAL_COUNT + VAL_TO_TEST_COUNT} val sequences, got {len(val_seq_dirs_all)}"
    )

if MAX_TRAIN_SEQS is not None:
    train_seq_dirs = train_seq_dirs[:MAX_TRAIN_SEQS]

val_seq_dirs = val_seq_dirs_all[:VAL_TO_VAL_COUNT]
test_seq_dirs = val_seq_dirs_all[VAL_TO_VAL_COUNT:VAL_TO_VAL_COUNT + VAL_TO_TEST_COUNT]

print(f"Train sequences used: {len(train_seq_dirs)}")
print(f"Val sequences used  : {len(val_seq_dirs)}")
print(f"Test sequences used : {len(test_seq_dirs)}")

if RESET_PROCESSED_DATASET and SPLIT_ROOT.exists():
    print("RESET_PROCESSED_DATASET=True -> deleting processed splits")
    shutil.rmtree(SPLIT_ROOT)

if FORCE_REBUILD_SPLITS and SPLIT_ROOT.exists():
    print("FORCE_REBUILD_SPLITS=True -> rebuilding processed splits")
    shutil.rmtree(SPLIT_ROOT)

if split_cache_ready(SPLIT_ROOT):
    print("Processed splits already exist. Reusing cache.")
else:
    print("Building processed splits at:", SPLIT_ROOT)
    SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

    for split_name, split_seq_dirs in [
        ("train", train_seq_dirs),
        ("val", val_seq_dirs),
        ("test", test_seq_dirs),
    ]:
        for seq_dir in split_seq_dirs:
            extract_from_reds_sequence(
                seq_dir=seq_dir,
                split_name=split_name,
                clip_name=seq_dir.name,
                scale=SCALE,
                max_frames=MAX_FRAMES_PER_SEQ,
            )

print("\nProcessed split summary:")
for split in ["train", "val", "test"]:
    split_dir = SPLIT_ROOT / split
    clips = [p for p in split_dir.iterdir() if p.is_dir()] if split_dir.exists() else []
    print(split, "clips:", len(clips))

Train sequences used: 240
Val sequences used  : 5
Test sequences used : 5
Processed splits already exist. Reusing cache.

Processed split summary:
train clips: 240
val clips: 5
test clips: 5


In [7]:
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))

print("Train dir exists:", TRAIN_SHARP_DIR.exists())
print("Val dir exists:", VAL_SHARP_DIR.exists())
print("Train png count:", len(list(TRAIN_SHARP_DIR.rglob("*.png"))))
print("Val png count:", len(list(VAL_SHARP_DIR.rglob("*.png"))))
# print("BEST_SOURCE_PATH exists:", BEST_SOURCE_PATH.exists())
print("SPLIT_ROOT exists:", SPLIT_ROOT.exists())


cuda available: True
gpu count: 1
gpu name: NVIDIA RTX PRO 6000 Blackwell Server Edition
Train dir exists: True
Val dir exists: True
Train png count: 24007
Val png count: 2965
SPLIT_ROOT exists: True


In [ ]:
# ── DataLoaders. Dataset and augment/tensor helpers are defined in earlier cells; fork-based workers inherit them from the parent process. ──
import random
import numpy as np
import cv2
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class CustomVideoDataset(Dataset):
    def __init__(self, split_root, seq_len=7, patch_size=64, training=True, scale=4, full_frame_eval=False):
        self.split_root = Path(split_root)
        self.seq_len = seq_len
        self.patch_size = patch_size
        self.training = training
        self.scale = scale
        self.full_frame_eval = full_frame_eval
        self.half = seq_len // 2

        self.samples = []
        self.clips = []

        clip_dirs = sorted([p for p in self.split_root.iterdir() if p.is_dir()])

        for clip_dir in clip_dirs:
            hr_dir = clip_dir / "hr_frames"
            lr_dir = clip_dir / "lr_frames"

            hr_frames = sorted(hr_dir.glob("*.png"))
            lr_frames = sorted(lr_dir.glob("*.png"))

            if len(hr_frames) == 0 or len(lr_frames) == 0:
                continue
            if len(hr_frames) != len(lr_frames):
                continue
            if len(hr_frames) < self.seq_len:
                continue

            self.clips.append(clip_dir)

            for center_idx in range(self.half, len(hr_frames) - self.half):
                lr_seq_paths = lr_frames[center_idx - self.half:center_idx + self.half + 1]
                hr_seq_paths = hr_frames[center_idx - self.half:center_idx + self.half + 1]

                self.samples.append({
                    "lr_seq_paths": lr_seq_paths,
                    "hr_seq_paths": hr_seq_paths,
                    "clip_name": clip_dir.name,
                    "center_idx": center_idx,
                })

        if len(self.samples) == 0:
            raise RuntimeError(f"No valid samples found in {self.split_root}")

        print(f"{self.split_root.name}: found {len(self.samples)} samples from {len(self.clips)} clips")

    def __len__(self):
        return len(self.samples)

    def _read_image(self, path):
        import cv2
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"Failed to read image: {path}")
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def _random_crop_sequences(self, lr_seq, hr_seq):
        h_lr, w_lr = lr_seq[0].shape[:2]
        lr_crop = self.patch_size
        hr_crop = self.patch_size * self.scale

        if h_lr < lr_crop or w_lr < lr_crop:
            raise RuntimeError(
                f"LR frame too small for crop: got {(h_lr, w_lr)}, need at least {(lr_crop, lr_crop)}"
            )

        top = random.randint(0, h_lr - lr_crop)
        left = random.randint(0, w_lr - lr_crop)

        lr_seq_crop = [img[top:top + lr_crop, left:left + lr_crop, :] for img in lr_seq]

        hr_top = top * self.scale
        hr_left = left * self.scale
        hr_seq_crop = [
            img[hr_top:hr_top + hr_crop, hr_left:hr_left + hr_crop, :]
            for img in hr_seq
        ]

        return lr_seq_crop, hr_seq_crop

    def _center_crop_sequences(self, lr_seq, hr_seq):
        h_lr, w_lr = lr_seq[0].shape[:2]
        lr_crop = min(self.patch_size, h_lr, w_lr)
        hr_crop = lr_crop * self.scale

        top = (h_lr - lr_crop) // 2
        left = (w_lr - lr_crop) // 2

        lr_seq_crop = [img[top:top + lr_crop, left:left + lr_crop, :] for img in lr_seq]

        hr_top = top * self.scale
        hr_left = left * self.scale
        hr_seq_crop = [
            img[hr_top:hr_top + hr_crop, hr_left:hr_left + hr_crop, :]
            for img in hr_seq
        ]

        return lr_seq_crop, hr_seq_crop

    def __getitem__(self, idx):
        sample = self.samples[idx]
        lr_seq = [self._read_image(p) for p in sample["lr_seq_paths"]]
        hr_seq = [self._read_image(p) for p in sample["hr_seq_paths"]]

        if self.training:
            lr_seq, hr_seq = self._random_crop_sequences(lr_seq, hr_seq)
            lr_seq, hr_seq = random_augment_sequences(lr_seq, hr_seq)
        else:
            if not self.full_frame_eval:
                lr_seq, hr_seq = self._center_crop_sequences(lr_seq, hr_seq)

        lr_seq = [img_to_tensor_rgb(img) for img in lr_seq]
        hr_seq = [img_to_tensor_rgb(img) for img in hr_seq]

        lr_seq = torch.stack(lr_seq, dim=0)
        hr_seq = torch.stack(hr_seq, dim=0)

        meta = {
            "clip_name": sample["clip_name"],
            "center_idx": sample["center_idx"],
        }
        return lr_seq, hr_seq, meta


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def eval_collate_fn(batch):
    lr_list, hr_list, meta_list = zip(*batch)
    lr_batch = torch.stack(lr_list, dim=0)
    hr_batch = torch.stack(hr_list, dim=0)
    return lr_batch, hr_batch, list(meta_list)

g = torch.Generator()
g.manual_seed(SEED)

_persistent = False
_prefetch = None
_mp_context = "fork" if NUM_WORKERS > 0 else None

train_dataset = CustomVideoDataset(
    SPLIT_ROOT / "train",
    seq_len=SEQ_LEN,
    patch_size=PATCH_SIZE,
    training=True,
    scale=SCALE,
    full_frame_eval=False
)

val_dataset = CustomVideoDataset(
    SPLIT_ROOT / "val",
    seq_len=SEQ_LEN,
    patch_size=PATCH_SIZE,
    training=False,
    scale=SCALE,
    full_frame_eval=FULL_FRAME_EVAL
)

test_dataset = CustomVideoDataset(
    SPLIT_ROOT / "test",
    seq_len=SEQ_LEN,
    patch_size=PATCH_SIZE,
    training=False,
    scale=SCALE,
    full_frame_eval=FULL_FRAME_EVAL
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    generator=g,
    collate_fn=eval_collate_fn,
    persistent_workers=_persistent,
    prefetch_factor=_prefetch,
    multiprocessing_context=_mp_context,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    generator=g,
    collate_fn=eval_collate_fn,
    persistent_workers=_persistent,
    prefetch_factor=_prefetch,
    multiprocessing_context=_mp_context,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    generator=g,
    collate_fn=eval_collate_fn,
    persistent_workers=_persistent,
    prefetch_factor=_prefetch,
    multiprocessing_context=_mp_context,
)

print("DataLoaders ready")
print("NUM_WORKERS:", NUM_WORKERS)
print("persistent_workers:", _persistent)

train: found 20640 samples from 240 clips
val: found 430 samples from 5 clips
test: found 430 samples from 5 clips
DataLoaders ready
NUM_WORKERS: 8
persistent_workers: False


In [9]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        identity = x
        out = self.act(self.conv1(x))
        out = self.conv2(out)
        return identity + out

class ConvResidualBlocks(nn.Module):
    def __init__(self, in_channels, out_channels, num_blocks):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.body = nn.Sequential(*[ResidualBlock(out_channels) for _ in range(num_blocks)])

    def forward(self, x):
        return self.body(self.head(x))


# ===================== SPyNet =====================
class SPyNetBasicModule(nn.Module):
    """A single level of SPyNet."""
    def __init__(self):
        super().__init__()
        self.basic_module = nn.ModuleList([
            self._make_layer(8, 32),
            self._make_layer(32, 64),
            self._make_layer(64, 32),
            self._make_layer(32, 16),
            self._make_layer(16, 2),
        ])

    def _make_layer(self, in_ch, out_ch):
        m = nn.Module()
        m.conv = nn.Conv2d(in_ch, out_ch, 7, 1, 3)
        return m

    def forward(self, tensor_input):
        x = tensor_input
        for i, layer in enumerate(self.basic_module):
            x = layer.conv(x)
            if i < len(self.basic_module) - 1:  # no ReLU on last layer
                x = F.relu(x, inplace=True)
        return x

class SPyNet(nn.Module):
    """SPyNet: Spatial Pyramid Network for optical flow estimation.
    Pretrained weights from: https://github.com/open-mmlab/mmediting
    """
    PRETRAINED_URL = (
        "https://download.openmmlab.com/mmediting/restorers/"
        "basicvsr/spynet_20210409-c6c1bd09.pth"
    )

    def __init__(self, pretrained=True):
        super().__init__()
        self.basic_module = nn.ModuleList([SPyNetBasicModule() for _ in range(6)])

        if pretrained:
            import urllib.request, tempfile, os
            weights_dir = os.path.join(tempfile.gettempdir(), "spynet_weights")
            os.makedirs(weights_dir, exist_ok=True)
            weights_path = os.path.join(weights_dir, "spynet_20210409-c6c1bd09.pth")
            if not os.path.exists(weights_path):
                print(f"Downloading SPyNet weights to {weights_path}...")
                urllib.request.urlretrieve(self.PRETRAINED_URL, weights_path)
                print("Download complete.")
            state_dict = torch.load(weights_path, map_location="cpu")
            self.load_state_dict(state_dict)
            print("Loaded pretrained SPyNet weights")

        self.register_buffer("mean", torch.Tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.Tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def preprocess(self, tensor_input):
        return (tensor_input - self.mean) / self.std

    def forward(self, ref, supp):
        """Estimate optical flow from ref to supp."""
        ref = [self.preprocess(ref)]
        supp = [self.preprocess(supp)]

        # build pyramid
        for _ in range(5):
            ref.insert(0, F.avg_pool2d(ref[0], kernel_size=2, stride=2, count_include_pad=False))
            supp.insert(0, F.avg_pool2d(supp[0], kernel_size=2, stride=2, count_include_pad=False))

        flow = ref[0].new_zeros(ref[0].shape[0], 2, ref[0].shape[2], ref[0].shape[3])

        for level in range(len(ref)):
            upsampled_flow = F.interpolate(flow, scale_factor=2, mode="bilinear", align_corners=True) * 2.0 if level > 0 else flow

            # pad if needed
            h, w = ref[level].shape[2:4]
            uh, uw = upsampled_flow.shape[2:4]
            if uh != h or uw != w:
                upsampled_flow = F.interpolate(upsampled_flow, size=(h, w), mode="bilinear", align_corners=True)

            # warp supp
            flow_for_warp = upsampled_flow.permute(0, 2, 3, 1)
            b, fh, fw, _ = flow_for_warp.shape
            yy, xx = torch.meshgrid(torch.arange(fh, device=flow.device, dtype=flow.dtype),
                                     torch.arange(fw, device=flow.device, dtype=flow.dtype), indexing="ij")
            grid = torch.stack((xx, yy), dim=-1).unsqueeze(0).expand(b, -1, -1, -1)
            vgrid = grid + flow_for_warp
            vgrid[..., 0] = 2.0 * vgrid[..., 0] / max(fw - 1, 1) - 1.0
            vgrid[..., 1] = 2.0 * vgrid[..., 1] / max(fh - 1, 1) - 1.0
            warped = F.grid_sample(supp[level], vgrid, mode="bilinear", padding_mode="border", align_corners=True)

            flow_input = torch.cat([ref[level], warped, upsampled_flow], dim=1)
            flow = upsampled_flow + self.basic_module[level](flow_input)

        return flow


# ===================== Flow Warp =====================
def flow_warp(x, flow):
    x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    flow = torch.nan_to_num(flow, nan=0.0, posinf=0.0, neginf=0.0)

    b, c, h, w = x.shape
    yy, xx = torch.meshgrid(torch.arange(h, device=x.device), torch.arange(w, device=x.device), indexing="ij")
    grid = torch.stack((xx, yy), dim=0).float().unsqueeze(0).repeat(b, 1, 1, 1)
    vgrid = grid + flow

    vgrid_x = 2.0 * vgrid[:, 0] / max(w - 1, 1) - 1.0
    vgrid_y = 2.0 * vgrid[:, 1] / max(h - 1, 1) - 1.0
    vgrid_x = torch.clamp(torch.nan_to_num(vgrid_x, nan=0.0, posinf=1.0, neginf=-1.0), -1.0, 1.0)
    vgrid_y = torch.clamp(torch.nan_to_num(vgrid_y, nan=0.0, posinf=1.0, neginf=-1.0), -1.0, 1.0)
    vgrid = torch.stack((vgrid_x, vgrid_y), dim=-1)

    out = F.grid_sample(x, vgrid, mode="bilinear", padding_mode="border", align_corners=True)
    return torch.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


# ===================== Upsampler =====================
class PixelShuffleUpsample(nn.Module):
    def __init__(self, num_feats, scale):
        super().__init__()
        if scale == 2:
            self.net = nn.Sequential(
                nn.Conv2d(num_feats, num_feats * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(num_feats, 3, 3, 1, 1),
            )
        elif scale == 4:
            self.net = nn.Sequential(
                nn.Conv2d(num_feats, num_feats * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(num_feats, num_feats * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(num_feats, 3, 3, 1, 1),
            )
        else:
            raise ValueError("Only scale=2 or scale=4 supported")

    def forward(self, x):
        return self.net(x)


# ===================== BasicVSR with SPyNet =====================
class BasicVSRRecurrentSeq(nn.Module):
    def __init__(self, seq_len=7, scale=4, num_feats=64, num_extract_blocks=5, num_prop_blocks=20, num_recon_blocks=5):
        super().__init__()
        if seq_len % 2 == 0:
            raise ValueError("seq_len must be odd")

        self.seq_len = seq_len
        self.scale = scale

        self.feat_extractor = ConvResidualBlocks(3, num_feats, num_extract_blocks)

        # SPyNet for optical flow (pretrained, fine-tuned with lower LR)
        self.flow_estimator = SPyNet(pretrained=True)

        self.backward_trunk = ConvResidualBlocks(num_feats * 2, num_feats, num_prop_blocks)
        self.forward_trunk = ConvResidualBlocks(num_feats * 2, num_feats, num_prop_blocks)

        fusion_layers = [
            nn.Conv2d(num_feats * 2, num_feats, 1, 1, 0),
            nn.LeakyReLU(0.1, inplace=True)
        ]
        for _ in range(num_recon_blocks):
            fusion_layers.append(ResidualBlock(num_feats))
        self.fusion = nn.Sequential(*fusion_layers)

        self.upsample = PixelShuffleUpsample(num_feats, scale)

    def compute_flows(self, x):
        b, t, c, h, w = x.shape
        flows_backward = [None] * (t - 1)
        flows_forward = [None] * (t - 1)

        for i in range(t - 1):
            flows_backward[i] = self.flow_estimator(x[:, i], x[:, i + 1])

        for i in range(1, t):
            flows_forward[i - 1] = self.flow_estimator(x[:, i], x[:, i - 1])

        return flows_forward, flows_backward

    def forward(self, x):
        b, t, c, h, w = x.shape
        feats_batch = self.feat_extractor(x.reshape(b * t, c, h, w))
        feats = list(feats_batch.reshape(b, t, -1, h, w).unbind(dim=1))
        flows_forward, flows_backward = self.compute_flows(x)

        backward_feats = [None] * t
        prop = torch.zeros_like(feats[0])

        for i in range(t - 1, -1, -1):
            if i < t - 1:
                prop = flow_warp(prop, flows_backward[i])
            prop = self.backward_trunk(torch.cat([feats[i], prop], dim=1))
            backward_feats[i] = prop

        forward_prop = torch.zeros_like(feats[0])
        outputs = []

        for i in range(t):
            if i > 0:
                forward_prop = flow_warp(forward_prop, flows_forward[i - 1])

            forward_prop = self.forward_trunk(torch.cat([feats[i], forward_prop], dim=1))
            fused = self.fusion(torch.cat([forward_prop, backward_feats[i]], dim=1))
            out = torch.clamp(self.upsample(fused), 0.0, 1.0)
            outputs.append(out)

        return torch.stack(outputs, dim=1)

model = BasicVSRRecurrentSeq(
    seq_len=SEQ_LEN,
    scale=SCALE,
    num_feats=64,
    num_extract_blocks=5,
    num_prop_blocks=20,
    num_recon_blocks=5
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"SPyNet params: {sum(p.numel() for p in model.flow_estimator.parameters()):,}")
print(model.__class__.__name__)


Download complete.
Loaded pretrained SPyNet weights
Total params: 5,587,887
Trainable params: 5,587,887
SPyNet params: 1,440,300
BasicVSRRecurrentSeq


In [10]:
import torchvision.models as models

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        diff = pred - target
        return torch.sqrt(diff * diff + self.eps * self.eps).mean()

class SobelEdgeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer("sobel_x", sobel_x)
        self.register_buffer("sobel_y", sobel_y)

    def _grad(self, x):
        sobel_x = self.sobel_x.to(device=x.device, dtype=x.dtype)
        sobel_y = self.sobel_y.to(device=x.device, dtype=x.dtype)
        grads = []
        for c in range(x.shape[1]):
            xc = x[:, c:c+1]
            gx = F.conv2d(xc, sobel_x, padding=1)
            gy = F.conv2d(xc, sobel_y, padding=1)
            grads.append(torch.sqrt(gx * gx + gy * gy + 1e-6))
        return torch.cat(grads, dim=1)

    def forward(self, pred, target):
        return F.l1_loss(self._grad(pred), self._grad(target))

class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
        self.slice1 = nn.Sequential(*[vgg[i] for i in range(9)])
        self.slice2 = nn.Sequential(*[vgg[i] for i in range(9, 27)])
        for param in self.parameters():
            param.requires_grad = False
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        pred = (pred - self.mean) / self.std
        target = (target - self.mean) / self.std
        pred_f1 = self.slice1(pred)
        target_f1 = self.slice1(target)
        pred_f2 = self.slice2(pred_f1)
        target_f2 = self.slice2(target_f1)
        return F.l1_loss(pred_f1, target_f1) + F.l1_loss(pred_f2, target_f2)

class CombinedRestorationLoss(nn.Module):
    def __init__(self, charbonnier_weight=1.0, edge_weight=0.05, perceptual_weight=0.1):
        super().__init__()
        self.charb = CharbonnierLoss(eps=1e-6)
        self.edge = SobelEdgeLoss()
        self.perceptual = VGGPerceptualLoss()
        self.charbonnier_weight = charbonnier_weight
        self.edge_weight = edge_weight
        self.perceptual_weight = perceptual_weight

    def forward(self, pred, target):
        charb_loss = self.charbonnier_weight * self.charb(pred, target)
        if pred.dim() == 5:
            b, t, c, h, w = pred.shape
            pred_4d = pred.reshape(b * t, c, h, w)
            target_4d = target.reshape(b * t, c, h, w)
        else:
            pred_4d = pred
            target_4d = target
        edge_loss = self.edge_weight * self.edge(pred_4d, target_4d)
        perceptual_loss = self.perceptual_weight * self.perceptual(pred_4d, target_4d)
        return charb_loss + edge_loss + perceptual_loss

criterion = CombinedRestorationLoss(
    charbonnier_weight=1.0,
    edge_weight=0.05,
    perceptual_weight=0.1,
)
criterion = criterion.to(device)

# Use lower LR for pretrained SPyNet (as per the paper)
spynet_params = list(model.flow_estimator.parameters())
spynet_ids = set(id(p) for p in spynet_params)
other_params = [p for p in model.parameters() if id(p) not in spynet_ids]

optimizer = optim.AdamW([
    {"params": other_params, "lr": LR},
    {"params": spynet_params, "lr": LR * 0.125},  # 2.5e-5 as per BasicVSR paper
], weight_decay=WEIGHT_DECAY)

scheduler = None
if USE_SCHEDULER:
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MIN_LR
    )

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP) if USE_AMP else None

print("Criterion:", criterion.__class__.__name__)
print("  - Charbonnier weight:", criterion.charbonnier_weight)
print("  - Edge weight:", criterion.edge_weight)
print("  - Perceptual weight:", criterion.perceptual_weight)
print("Optimizer:", optimizer.__class__.__name__)
print("  - Main LR:", LR)
print("  - SPyNet LR:", LR * 0.125)
print("Scheduler:", scheduler.__class__.__name__ if scheduler is not None else None)
print("AMP enabled:", USE_AMP)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:06<00:00, 85.9MB/s] 


Criterion: CombinedRestorationLoss
  - Charbonnier weight: 1.0
  - Edge weight: 0.05
  - Perceptual weight: 0.1
Optimizer: AdamW
  - Main LR: 0.0002
  - SPyNet LR: 2.5e-05
Scheduler: ReduceLROnPlateau
AMP enabled: False


In [11]:
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None, grad_clip_norm=None):
    model.train()
    running_loss = 0.0
    valid_batches = 0
    skipped_batches = 0
    amp_device = "cuda" if torch.cuda.is_available() else "cpu"

    for lr_seq, hr_seq, _ in tqdm(loader, desc="Train", leave=False):
        lr_seq = lr_seq.to(device, non_blocking=True)
        hr_seq = hr_seq.to(device, non_blocking=True)

        if not torch.isfinite(lr_seq).all() or not torch.isfinite(hr_seq).all():
            skipped_batches += 1
            continue

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None and USE_AMP:
            with torch.amp.autocast(amp_device, enabled=True):
                pred = model(lr_seq)
                if not torch.isfinite(pred).all():
                    skipped_batches += 1
                    continue

                loss = criterion(pred, hr_seq)
                if not torch.isfinite(loss):
                    skipped_batches += 1
                    continue

            scaler.scale(loss).backward()

            if grad_clip_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(lr_seq)
            if not torch.isfinite(pred).all():
                skipped_batches += 1
                continue

            loss = criterion(pred, hr_seq)
            if not torch.isfinite(loss):
                skipped_batches += 1
                continue

            loss.backward()

            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            optimizer.step()

        running_loss += loss.item()
        valid_batches += 1

    if skipped_batches > 0:
        print(f"Skipped {skipped_batches} non-finite train batches")

    if valid_batches == 0:
        return float("nan")

    return running_loss / valid_batches

@torch.no_grad()
def evaluate_model(model, loader, criterion, device, scale=4, shave_border=0, max_batches=None, compute_ssim=True):
    model.eval()
    total_loss = 0.0
    total_psnr = 0.0
    total_ssim = 0.0
    total_bicubic_psnr = 0.0
    total_bicubic_ssim = 0.0
    count = 0
    ssim_count = 0           # <<< FIX: separate counter for SSIM
    bicubic_ssim_count = 0   # <<< FIX: separate counter for bicubic SSIM
    per_clip = {}

    for batch_idx, (lr_seq, hr_seq, meta_list) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        lr_seq = lr_seq.to(device, non_blocking=True)
        hr_seq = hr_seq.to(device, non_blocking=True)

        if not torch.isfinite(lr_seq).all() or not torch.isfinite(hr_seq).all():
            continue

        pred = model(lr_seq)
        if not torch.isfinite(pred).all():
            continue

        loss = criterion(pred, hr_seq)
        if not torch.isfinite(loss):
            continue

        pred_psnr = calc_psnr(pred, hr_seq, shave_border=shave_border)
        pred_ssim = calc_ssim_sequence(pred, hr_seq, shave_border=shave_border) if compute_ssim else None

        bicubic = bicubic_upsample_sequence(lr_seq, scale=scale)
        bicubic_psnr = calc_psnr(bicubic, hr_seq, shave_border=shave_border)
        bicubic_ssim = calc_ssim_sequence(bicubic, hr_seq, shave_border=shave_border) if compute_ssim else None

        if not math.isfinite(pred_psnr) or not math.isfinite(bicubic_psnr):
            continue

        total_loss += loss.item()
        total_psnr += pred_psnr
        total_bicubic_psnr += bicubic_psnr
        if pred_ssim is not None:
            total_ssim += pred_ssim
            ssim_count += 1              # <<< FIX
        if bicubic_ssim is not None:
            total_bicubic_ssim += bicubic_ssim
            bicubic_ssim_count += 1      # <<< FIX

        meta = meta_list[0]
        clip_name = meta["clip_name"]

        if clip_name not in per_clip:
            per_clip[clip_name] = {
                "count": 0,
                "loss": 0.0,
                "psnr": 0.0,
                "ssim": 0.0,
                "ssim_count": 0,             # <<< FIX
                "bicubic_psnr": 0.0,
                "bicubic_ssim": 0.0,
                "bicubic_ssim_count": 0,     # <<< FIX
            }

        per_clip[clip_name]["count"] += 1
        per_clip[clip_name]["loss"] += loss.item()
        per_clip[clip_name]["psnr"] += pred_psnr
        per_clip[clip_name]["bicubic_psnr"] += bicubic_psnr

        if pred_ssim is not None:
            per_clip[clip_name]["ssim"] += pred_ssim
            per_clip[clip_name]["ssim_count"] += 1           # <<< FIX
        if bicubic_ssim is not None:
            per_clip[clip_name]["bicubic_ssim"] += bicubic_ssim
            per_clip[clip_name]["bicubic_ssim_count"] += 1   # <<< FIX

        count += 1

    if count == 0:
        return {
            "loss": float("nan"),
            "psnr": float("nan"),
            "ssim": None,
            "bicubic_psnr": float("nan"),
            "bicubic_ssim": None,
            "per_clip": {}
        }

    for clip_name in per_clip:
        c = per_clip[clip_name]["count"]
        per_clip[clip_name]["loss"] /= c
        per_clip[clip_name]["psnr"] /= c
        per_clip[clip_name]["bicubic_psnr"] /= c
        # <<< FIX: use per-clip ssim_count instead of c
        sc = per_clip[clip_name]["ssim_count"]
        per_clip[clip_name]["ssim"] = (per_clip[clip_name]["ssim"] / sc) if sc > 0 else None
        bsc = per_clip[clip_name]["bicubic_ssim_count"]
        per_clip[clip_name]["bicubic_ssim"] = (per_clip[clip_name]["bicubic_ssim"] / bsc) if bsc > 0 else None

    return {
        "loss": total_loss / count,
        "psnr": total_psnr / count,
        "ssim": (total_ssim / ssim_count) if ssim_count > 0 else None,                    # <<< FIX
        "bicubic_psnr": total_bicubic_psnr / count,
        "bicubic_ssim": (total_bicubic_ssim / bicubic_ssim_count) if bicubic_ssim_count > 0 else None,  # <<< FIX
        "per_clip": per_clip,
    }

@torch.no_grad()
def benchmark_model_latency(model, loader, device, warmup=3, max_iters=20):
    model.eval()
    batches = []
    for i, batch in enumerate(loader):
        if i >= max_iters:
            break
        batches.append(batch)

    for i in range(min(warmup, len(batches))):
        _ = model(batches[i][0].to(device, non_blocking=True))

    if device.type == "cuda":
        torch.cuda.synchronize()

    times = []
    for batch in batches:
        lr_seq = batch[0].to(device, non_blocking=True)
        start = time.time()
        _ = model(lr_seq)
        if device.type == "cuda":
            torch.cuda.synchronize()
        end = time.time()
        times.append(end - start)

    if len(times) == 0:
        return None

    avg_time = float(np.mean(times))
    return {
        "avg_seconds_per_sequence": avg_time,
        "sequences_per_second": (1.0 / avg_time) if avg_time > 0 else 0.0
    }

class EarlyStoppingFull:
    def __init__(self, patience_full=8, min_delta=1e-4, min_epochs=100):
        self.patience_full = patience_full
        self.min_delta = min_delta
        self.min_epochs = min_epochs
        self.best = None
        self.num_bad_full_checks = 0

    def step(self, current_full_metric, current_epoch):
        if current_epoch < self.min_epochs:
            return False
        if self.best is None or current_full_metric < self.best - self.min_delta:
            self.best = current_full_metric
            self.num_bad_full_checks = 0
            return False
        self.num_bad_full_checks += 1
        return self.num_bad_full_checks >= self.patience_full

print("Training helpers ready")

Training helpers ready


In [12]:
if not Path(HISTORY_JSON_PATH).exists():
    raise FileNotFoundError(
        f"History file not found: {HISTORY_JSON_PATH}. "
        "Run the training cell first \u2014 this cell only visualizes an existing training history."
    )

with open(HISTORY_JSON_PATH, "r") as f:
    history = json.load(f)

for i in range(len(history["train_loss"])):
    epoch = i + 1
    mode = history["val_mode"][i] if "val_mode" in history else "?"
    ssim = history["val_ssim"][i]
    ssim_str = f"{ssim:.4f}" if ssim is not None else "N/A"
    print(
        f"Epoch {epoch:03d} | mode={mode} | "
        f"train_loss={history['train_loss'][i]:.4f} | "
        f"val_loss={history['val_loss'][i]:.4f} | "
        f"val_psnr={history['val_psnr'][i]:.2f} | "
        f"val_ssim={ssim_str} | "
        f"lr={history['lr'][i]:.2e}"
    )

    # Find best FULL epochs
full_psnrs = [(i+1, history["val_psnr"][i]) for i in range(len(history["val_psnr"])) 
              if history["val_mode"][i] == "FULL"]
best_psnr_epoch, best_psnr = max(full_psnrs, key=lambda x: x[1])

full_losses = [(i+1, history["val_loss"][i]) for i in range(len(history["val_loss"])) 
               if history["val_mode"][i] == "FULL"]
best_loss_epoch, best_loss = min(full_losses, key=lambda x: x[1])

print(f"\n--- BEST RESULTS ---")
print(f"Best PSNR: {best_psnr:.2f} dB at epoch {best_psnr_epoch}")
print(f"Best Loss: {best_loss:.4f} at epoch {best_loss_epoch}")

Epoch 001 | mode=FAST | train_loss=0.5326 | val_loss=0.8742 | val_psnr=4.49 | val_ssim=N/A | lr=2.00e-04
Epoch 002 | mode=FULL | train_loss=0.1400 | val_loss=0.1248 | val_psnr=26.58 | val_ssim=N/A | lr=2.00e-04
Epoch 003 | mode=FAST | train_loss=0.1136 | val_loss=0.1318 | val_psnr=26.31 | val_ssim=N/A | lr=2.00e-04
Epoch 004 | mode=FULL | train_loss=0.1087 | val_loss=0.1105 | val_psnr=27.88 | val_ssim=N/A | lr=2.00e-04
Epoch 005 | mode=FAST | train_loss=0.1062 | val_loss=0.1257 | val_psnr=26.88 | val_ssim=N/A | lr=2.00e-04
Epoch 006 | mode=FULL | train_loss=0.1042 | val_loss=0.1066 | val_psnr=28.17 | val_ssim=N/A | lr=2.00e-04
Epoch 007 | mode=FAST | train_loss=0.1032 | val_loss=0.1238 | val_psnr=26.97 | val_ssim=N/A | lr=2.00e-04
Epoch 008 | mode=FULL | train_loss=0.1023 | val_loss=0.1046 | val_psnr=28.30 | val_ssim=N/A | lr=2.00e-04
Epoch 009 | mode=FAST | train_loss=0.1014 | val_loss=0.1210 | val_psnr=27.22 | val_ssim=N/A | lr=2.00e-04
Epoch 010 | mode=FULL | train_loss=0.1008 | val